# 02 — UC2 FedAvg

Train FedAvg (Federated Averaging) across the alpha sweep.
This is the standard FL baseline — full model exchange each round.

Paper reference: McMahan et al. "Communication-Efficient Learning 
of Deep Networks from Decentralized Data" (2017)

In [ ]:
import sys, os
sys.path.append("..")
import UC2Utils as uc2
sys.path.insert(0, uc2.LIB_DIR)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pickle
from collections import Counter


sys.path.insert(0, uc2.LIB_DIR)

print(f"Device: {uc2.DEFAULT_CONFIG['device']}")

## Configuration

In [ ]:
ALPHAS = [0.5, 1.0, 5.0, 10.0]
MODEL = "lstm"

OVERRIDES = dict(
    num_glob_iters=100,
    local_epochs=50,
    num_users=20,        # all 20 users participate each round
    early_stopping_patience=50,
)

## Train FedAvg for Each α

Lower α → more heterogeneous data across users → harder for FedAvg.
We expect degradation at low α since FedAvg assumes roughly IID data.

In [ ]:
results_fedavg = {}

for alpha in ALPHAS:
    print(f"\n{'='*60}")
    print(f"  FedAvg — α={alpha}")
    print(f"{'='*60}")
    
    try:
        server, result = uc2.run_experiment(
            algorithm="FedAvg",
            alpha=alpha,
            **OVERRIDES
        )
        results_fedavg[alpha] = result
        
        # Print final metrics
        glob_metrics = result["metrics"]["glob_test_metric"]
        if glob_metrics:
            best_idx = np.argmin([m.get("unscaled_mae", float("inf")) 
                                  for m in glob_metrics])
            best = glob_metrics[best_idx]
            print(f"\n  Best round: {best_idx}")
            print(f"  MAE (scaled):   {best.get('mae', 'N/A'):.4f}")
            print(f"  MAE (unscaled): {best.get('unscaled_mae', 'N/A'):.4f}")
            print(f"  MAPE:           {best.get('unscaled_mape', 'N/A'):.4f}")
        
        # Per-user breakdown
        per_user = uc2.evaluate_server(server)
        per_user_mae = per_user["metrics"].get("unscaled_mae", [])
        if per_user_mae:
            print(f"\n  Per-user MAE: mean={np.mean(per_user_mae):.4f}, "
                  f"std={np.std(per_user_mae):.4f}, "
                  f"CV={np.std(per_user_mae)/np.mean(per_user_mae):.3f}")
            
    except Exception as e:
        print(f"  [ERROR] {e}")
        import traceback
        traceback.print_exc()

## Training Curves

In [ ]:
import matplotlib.pyplot as plt
uc2.setup_plot_style()

fig, ax = plt.subplots(figsize=(10, 5))
for alpha, r in sorted(results_fedavg.items()):
    glob_metrics = r["metrics"].get("glob_test_metric", [])
    maes = [m.get("unscaled_mae") for m in glob_metrics]
    ax.plot(maes, label=f"α={alpha}", linewidth=2)

ax.set_xlabel("Communication Round")
ax.set_ylabel("Unscaled MAE (MB)")
ax.set_title("FedAvg Training Convergence per α")
ax.legend()
plt.tight_layout()
plt.show()

## Per-User Performance at Each α

In [ ]:
fig, axes = plt.subplots(1, len(ALPHAS), figsize=(5*len(ALPHAS), 5), sharey=True)

for ax, alpha in zip(axes, ALPHAS):
    if alpha not in results_fedavg:
        continue
    r = results_fedavg[alpha]
    per_user_mae = r.get("per_user", {}).get("metrics", {}).get("unscaled_mae", [])
    if per_user_mae:
        ax.bar(range(len(per_user_mae)), sorted(per_user_mae, reverse=True),
               color=uc2.COLORS["FedAvg"], alpha=0.7)
    ax.set_title(f"α = {alpha}")
    ax.set_xlabel("User (sorted)")
    if ax == axes[0]:
        ax.set_ylabel("Unscaled MAE")

plt.suptitle("FedAvg — Per-User MAE by Heterogeneity Level", fontsize=14)
plt.tight_layout()
plt.show()

## Communication Cost

FedAvg: C = 2|θ| × R × |S| where |θ|=3.7MB, R=rounds, |S|=20

In [ ]:
for alpha, r in sorted(results_fedavg.items()):
    n_rounds = r["n_rounds"]
    cost = uc2.comm_cost_fedavg(n_rounds, n_users_per_round=20)
    print(f"α={alpha}: {n_rounds} rounds → {cost:.1f} MB communication")

## Summary

FedAvg results saved to `results/fedavg/alpha_{α}/lstm/rep_0/`.
Key observations:
- Lower α (more heterogeneity) → higher MAE (expected)
- Full model exchange each round → high communication cost
- Per-user variance increases with heterogeneity